# Close 15 · Integration mapping

**Day 4 · Close** · 15 min · Worksheet, one per group · Runs on: Colab or a laptop, CPU only · No API key, no model calls, no installs, nothing read from the repo

**Follows** S25, which showed what a stranger's text does to a loop you built the day before.
**Hands off to** Day 5, all of it: S26 builds an MCP server in front of a system that is on the sheet you are about to fill, S27 and S28 assemble the capstone around those rows, and S29 turns the last three columns into the governance pack.

Everything this week ran against two servers, a JSON file and a folder of markdown. That was deliberate — it kept the variable in the lab. On Thursday the same loop has to reach a real ticketing system, a real document library and a real API, and every one of those has an owner, a credential, a change window and a person who can say no.

**This is the fifteen minutes where a lab becomes an integration.** One row per system your capstone touches. Not per tool: S24's blast radius sheet was per tool, and it stays per tool. This one is per *system*, because a system is the thing that has an owner, a network path, a data classification and an outage.

| § | What you do | The question it settles |
|---|---|---|
| 2 | read the worked example — this week's own stack, mapped | what a row has to say to be worth writing down |
| 3 | fill your own: four rows, eight minutes | what your capstone actually touches |
| 4 | run the four checks | which rows are a design problem rather than an access problem |
| 5 | read off the ask list and the build list | who you have to ask, and what you are mocking on Thursday |
| 6 | write the sheet out | the artifact you carry into S27 |

**The honest frame.** You are not getting production access to any of these by Thursday morning, and nobody expects you to. That is exactly why the sheet is worth fifteen minutes: it decides what you **mock** so the capstone runs, and what must be true before it is real. A mock you chose, with a row that says what it stands in for and who has to sign it off, is engineering. A mock you fell into at 11:00 on Thursday is a demo.


## 1. Setup

One cell, and it needs nothing — not the repo, not a key, not a package. This is the last notebook of the day and it must not be the one that fails at 15:00, so everything below is the Python standard library: no pandas, no tabulate, no install cell, nothing to go wrong on the one laptop that has been going wrong all week. It runs on a bare kernel, and it runs the same in Colab. If it cannot find the lab folder it writes next to wherever you opened it.


In [ ]:
# Setup: standard library only. No model, no network, no corpus, no third-party package.
from pathlib import Path

CELL = 52  # printed columns are truncated at this width. The written sheet keeps the full text


def show(rows, cols):
    """Rows of dicts as a fixed-width table. Nine lines, and it is why this notebook needs no pandas."""
    def cell(r, c):
        v = str(r.get(c, ""))
        return v if len(v) <= CELL else v[:CELL - 1] + "…"
    w = {c: max([len(c)] + [len(cell(r, c)) for r in rows]) for c in cols}
    print("  ".join(c.ljust(w[c]) for c in cols))
    print("  ".join("-" * w[c] for c in cols))
    for r in rows:
        print("  ".join(cell(r, c).ljust(w[c]) for c in cols))


def as_table(rows, cols):
    """The same rows as a markdown table, for the file this notebook writes."""
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(r.get(c, "")).replace("|", r"\|") for c in cols) + " |" for r in rows]
    return "\n".join(out)


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "services" / "mcp_servers").exists() or (p / "corpus").exists():
            return p
    return Path.cwd()


ROOT = find_root()
OUT = ROOT / "outputs" / "15_integration_mapping"
OUT.mkdir(parents=True, exist_ok=True)

GROUP = "group-?"                                    # TODO your group. It names the file you carry to Day 5
CAPSTONE = "{your capstone use case, one line}"      # TODO the brief you picked on Day 1, S5

print("writing to:", OUT)


## 2. A row, and what it has to say

Twelve fields. Most of them are one word, two of them are one line, and every one of them is there because a session this week went wrong without it.

| Field | What goes in it | Why it is on the sheet |
|---|---|---|
| `system` | the name people at OQ use for it, not the vendor's product name | you are going to say this name in a meeting with the team that runs it |
| `role` | system of record, document source, index, identity, notification, the model | it sets what a wrong answer costs, which is not the same for a manual and a ticket |
| `direction` | `read`, `write`, `read+write`, or `call` for a system that holds none of your records | a write is a different conversation, a different sign-off and a different Monday |
| `reach` | how you call it **today**: REST, MCP server, ODBC, a nightly CSV, a person emailing an export | "we'll use their API" is not a reach until somebody in the room has called it once |
| `access` | `have`, `requested`, `none` | the line between a plan and a wish, and it is the column that slips |
| `owner` | the person who can say yes. A name | a team cannot approve anything. "IT" has never signed off a thing |
| `authority` | which records, which states, how many. S24's policy, in one line | blank here means the prompt is your access control |
| `trust` | `trusted` or `untrusted`: did anyone outside your team write text that lands in this system | S25's first column, and the answer is `untrusted` more often than a room expects |
| `egress` | `none`, `internal`, `external`: can content leave through this hop **in a shape the model chooses** | S25's second column. It is what turns two problems into a channel |
| `session` | which session this row belongs to. `main` until you split one | splitting a session is a field on a sheet, not a discussion. §4 makes you use it |
| `day5` | `real`, `mock`, `out` | what Thursday morning actually contains |
| `blocker` | the one thing that has to be true before this row is real | this sentence is your answer when somebody asks what you need |

**Two notes before you read the example, because both catch people out.**

**The model endpoint is a row.** Everything you send it leaves OQ. That is a contract and a data classification question, it is real, and it belongs in `blocker` where the person who signs it can see it.

**Its `egress` is still `none`.** That column asks a narrower question: can a stranger's text, sitting in your context window, decide what goes out through this hop and where to? Your model call does not let it — you assemble that payload. A third-party tool with a free-text argument does, which is the whole of S25 §6 in one cell of one row.


In [ ]:
COLUMNS = ["system", "role", "direction", "reach", "access", "owner",
           "authority", "trust", "egress", "session", "day5", "blocker"]
MAP_VIEW = ["system", "role", "direction", "reach", "access", "owner", "day5", "blocker"]
TRUST_VIEW = ["system", "direction", "authority", "trust", "egress", "session"]

# This week's own stack, mapped. Four rows, and four shapes most capstones turn out to be.
EXAMPLE = [
    dict(system="SGP service desk", role="system of record", direction="read+write",
         reach="MCP server, 4 tools", access="have", owner="desk lead (in the lab: nobody)",
         authority="3 named tickets, open -> in_progress, max 2 writes",
         trust="untrusted", egress="internal", session="main", day5="real",
         blocker="none. It is a JSON file we own"),
    dict(system="SGP document store", role="document source", direction="read",
         reach="MCP server, BM25 over 74 files", access="have", owner="HSE document controller",
         authority="read-only, whole corpus",
         trust="untrusted", egress="none", session="main", day5="real",
         blocker="the real library has per-folder permissions"),
    dict(system="sgp-owner-lookup", role="third-party enrichment", direction="read",
         reach="registry install, version not pinned", access="have",
         owner="nobody at OQ", authority="none declared. It says read-only",
         trust="trusted", egress="external", session="main", day5="mock",
         blocker="a named owner and a pinned version"),
    dict(system="Model API (vendor)", role="the model", direction="call",
         reach="HTTPS, one key in .env", access="have", owner="whoever signed the vendor terms",
         authority="n/a", trust="trusted", egress="none", session="main", day5="real",
         blocker="data classification sign-off, and a named key owner"),
]


print("THE MAP\n")
show(EXAMPLE, MAP_VIEW)
print("\n\nTHE TRUST BOUNDARY\n")
show(EXAMPLE, TRUST_VIEW)


Three cells in that table are the whole of Day 4, and it is worth pointing at each one before you write your own.

- **`authority` on row one** is S24's policy, compressed to one line a person can sign. Everything that lab measured came from that line existing. A group that leaves it blank has not decided to allow everything — it has decided not to decide, and the prompt inherits the job.
- **`egress: external` on row three** is S25 §6. That server was useful, it declared itself read-only, the gate never fired, and a couple of hundred characters of the plant's ticket text left the building through an argument called `context`. It is still in the build here, at `day5: mock`, because that is how it would reach Thursday if nobody looked. §4 is what looks.
- **`blocker` on row four** is the row that stops a project in week six when it was not written down in week one. The model works, the demo lands, and then someone in legal asks where the ticket text goes. Write the sentence now, while it costs nothing.


## 3. Your map

Eight minutes. Four rows. Mostly one word per cell.

**Rules that keep it honest:**

- **One row per system.** Four is the right number. If you are at seven, two of them are the same system reached two ways, and one of them is not in scope this week.
- **Start with the system of record** — the thing whose state your capstone changes. If your capstone does not have one, say that out loud now: it changes the rung, and S20's table says so.
- **`owner` is a person.** If you cannot name one, write `?`. It is not a failure, it is the first line of your ask list, and §5 prints it for you.
- **`reach` is how you call it today,** not how the integration roadmap says you will in Q2.
- **Do not argue about `trust`.** If a contractor, a vendor, a caller or a scanner puts text into that system, it is `untrusted`. Almost everything at a plant is.
- **`blocker` last,** one line, the single thing that has to be true. If you write three things, you have not found the one.

Fill the cell below and run the one after it. Anything still `TODO` gets printed back at you, and any value the checks will not understand gets flagged before it can hide a finding.


In [ ]:
# TODO: your capstone. Replace every TODO. Row 4 is filled in because every group has it.
YOURS = [
    dict(system="TODO", role="system of record", direction="TODO", reach="TODO", access="TODO",
         owner="TODO", authority="TODO", trust="TODO", egress="TODO", session="main",
         day5="TODO", blocker="TODO"),
    dict(system="TODO", role="document source", direction="read", reach="TODO", access="TODO",
         owner="TODO", authority="read-only", trust="TODO", egress="none", session="main",
         day5="TODO", blocker="TODO"),
    dict(system="TODO", role="TODO", direction="TODO", reach="TODO", access="TODO",
         owner="TODO", authority="TODO", trust="TODO", egress="TODO", session="main",
         day5="TODO", blocker="TODO"),
    dict(system="Model API (vendor)", role="the model", direction="call",
         reach="HTTPS, one key held by the team", access="have", owner="TODO",
         authority="n/a", trust="trusted", egress="none", session="main", day5="real",
         blocker="TODO — who signs off that this text may leave OQ"),
]


In [ ]:
ALLOWED = {"direction": {"read", "write", "read+write", "call"},
           "access": {"have", "requested", "none"},
           "trust": {"trusted", "untrusted"},
           "egress": {"none", "internal", "external"},
           "day5": {"real", "mock", "out"}}


def audit_fill(rows, label):
    blanks = [(i + 1, k) for i, r in enumerate(rows) for k in COLUMNS
              if str(r.get(k, "")).strip().upper().startswith("TODO") or not str(r.get(k, "")).strip()]
    typos = [(i + 1, k, r[k]) for i, r in enumerate(rows) for k in ALLOWED
             if r.get(k) not in ALLOWED[k] and not str(r.get(k, "")).strip().upper().startswith("TODO")]
    print(f"{label}: {len(rows)} rows, {len(blanks)} cells still open, {len(typos)} values the checks cannot read\n")
    for row, field in blanks:
        print(f"   row {row}  {field}")
    for row, field, value in typos:
        print(f"   row {row}  {field}: '{value}' is not one of {sorted(ALLOWED[field])}")
    return not blanks and not typos


show(YOURS, MAP_VIEW)
print()
show(YOURS, TRUST_VIEW)
print()
READY = audit_fill(YOURS, "your sheet")


## 4. Four checks

The sheet is not a form. Four things fall straight out of it, and each one is a finding you would otherwise make on Thursday afternoon with the room watching.

| # | Check | The rule behind it | The fix |
|---|---|---|---|
| 1 | **the triangle** | untrusted text in, something worth reading, and a model-chosen way out — all three in one session is a channel, not three problems | change `session` on one row. Not the prompt |
| 2 | **the unbounded write** | a row that writes with no authority line. S24 measured what that costs in one shift | write which records, which states, how many |
| 3 | **the fiction** | `day5: real` on a system you do not have access to | make it `mock`, and say in `blocker` what the mock stands in for |
| 4 | **the read-only egress** | a read-only hop that can still carry content out. `read_only_hint` is a claim made by the party you would be defending against | allowlist it in your own config, pin the version, read what its arguments carry |

Rows marked `day5: out` are excluded — documenting a system you decided not to touch is the point of writing it down, and it should not keep firing at you.


In [ ]:
def checks(rows):
    live = [r for r in rows if r.get("day5") != "out"]
    found = []
    for s in sorted({r.get("session", "main") for r in live}):
        rs = [r for r in live if r.get("session", "main") == s]
        untrusted = [r["system"] for r in rs if r.get("trust") == "untrusted"]
        readable = [r["system"] for r in rs if "read" in str(r.get("direction"))]
        ways_out = [r["system"] for r in rs if r.get("egress") == "external"]
        if untrusted and readable and ways_out:
            found.append(("triangle", f"session '{s}' holds all three: untrusted text from "
                          f"{', '.join(untrusted)}; something worth reading in {', '.join(readable)}; "
                          f"a model-chosen way out through {', '.join(ways_out)}. Split it — put the "
                          f"egress row in its own session with no access to the rest."))
    for r in live:
        if "write" in str(r.get("direction")) and str(r.get("authority", "")).strip().lower() in {"", "n/a", "-", "none", "todo"}:
            found.append(("unbounded write", f"{r['system']} writes with no authority line. Which records, "
                          f"which states, how many — or the prompt is your access control."))
        if r.get("day5") == "real" and r.get("access") != "have":
            found.append(("fiction", f"{r['system']} is marked real for Thursday and access is "
                          f"'{r.get('access')}'. Make it a mock and say what the mock stands in for."))
        if r.get("egress") == "external" and "write" not in str(r.get("direction")):
            found.append(("read-only egress", f"{r['system']} is read-only and still a way out. Allowlist it "
                          f"in your config, pin the version, and read what its arguments carry."))
    return found


def report(rows, label):
    found = checks(rows)
    print(f"{label}: {len(found)} finding(s)")
    for i, (rule, text) in enumerate(found, 1):
        print(f"\n  {i}. [{rule}]  {text}")
    print()


report(EXAMPLE, "the worked example")
fixed = [dict(r, day5="out") if r["system"] == "sgp-owner-lookup" else r for r in EXAMPLE]
print(f"Change one field — sgp-owner-lookup to day5 'out' — and the same sheet reports "
      f"{len(checks(fixed))} findings.\nThat is the shape of the fix: a row leaves, not a prompt improves.\n")
print("-" * 100 + "\n")
report(YOURS, "your sheet")
if not checks(YOURS):
    print("Nothing fired. On a sheet that is not finished that means the sheet is not finished,",
          "\nnot that the design is clean. Finish §3 and run this again.")


The example fires twice, and both findings are the same row. `sgp-owner-lookup` is the only model-chosen way out of that session, so it is the third property of the triangle and it is the read-only egress. Take it out of the build and both clear at once — that is the printed line under the findings, and it is the whole argument of S25 §5 in one edit.

Two things worth saying while that output is on screen. The fix was a row leaving, not a prompt improving, and it cost nothing because it happened on a sheet rather than in a sprint. And the row does not disappear: `out` keeps it visible, with the reason it was refused, which is what you want six months from now when somebody proposes it again.

**What the checks do not tell you.** They read what you wrote. A system you forgot is not a finding, it is an incident in March. Before you move on, two questions round the group:

- **What reads what you write?** A note your loop puts on a ticket is read by a shift engineer at 03:00 who will act on it. That is an `egress: internal` you may not have ticked.
- **What else lands in the untrusted systems?** Email into the desk, uploads into the document library, scanned pages, a vendor's work order, another agent's output. S25's first defence was to list the channels, and most teams find more than they expected.


## 5. The ask list and the build list

Two lists fall out of the sheet with no further thinking, which is the reason for having filled it in rather than discussed it.

The **ask list** is the only part of Day 4 that has a date on it: a named person, one sentence about what you need from them, and when you need it by. It is also the first page of the governance pack you assemble in S29.

The **build list** is what Thursday morning actually contains: what is real, what is mocked and what each mock stands in for. S26 builds one MCP server in front of a mock ERP, as a class, and the point of that hour is that you can then do it again for the row below it.


In [ ]:
def ask_list(rows):
    asks = []
    for r in rows:
        if r.get("day5") == "out":
            continue
        if "?" in str(r.get("owner", "")) or str(r.get("owner", "")).strip().upper().startswith("TODO"):
            asks.append((r["system"], "?", "name the person who owns it"))
        if r.get("access") in {"requested", "none"}:
            asks.append((r["system"], r.get("owner", "?"), f"access: {r.get('reach')} ({r.get('access')} today)"))
        if "write" in str(r.get("direction")):
            asks.append((r["system"], r.get("owner", "?"), f"sign-off on the write policy: {r.get('authority')}"))
    return asks


print("THE ASK LIST — one line each, and every one of them needs a date before Thursday\n")
for system, owner, need in ask_list(YOURS):
    print(f"   {system:<28} {owner:<26} {need}")

print("\n\nTHE BUILD LIST\n")
for stance in ("real", "mock", "out"):
    rows = [r for r in YOURS if r.get("day5") == stance]
    print(f"   {stance:<5} ({len(rows)})")
    for r in rows:
        note = r.get("blocker") if stance != "out" else "documented, not touched"
        print(f"         {r['system']:<28} {note}")
unset = [r for r in YOURS if r.get("day5") not in {"real", "mock", "out"}]
if unset:
    print(f"   and {len(unset)} row(s) with no stance yet: " + ", ".join(r["system"] for r in unset))

candidates = [r for r in YOURS if r.get("day5") == "mock" and "write" in str(r.get("direction"))]
if candidates:
    print(f"\nS26 builds an MCP server in front of a mock system. The row it maps to on your sheet: "
          f"{candidates[0]['system']}.")


## 6. Write it down

The next cell writes two files into `outputs/15_integration_mapping/`.

**Your sheet**, filled, with the findings and both lists in it. Bring it to S27 — it is the input to the capstone scaffold, and the file the group argues from rather than re-deriving on Thursday.

**A blank worksheet**, because half a room would rather do this on paper the second time, and because the sheet is worth re-filling when your capstone changes shape on Day 5.


In [ ]:
def slug(text):
    keep = [c.lower() if c.isalnum() else "-" for c in text.strip()]
    return "".join(keep).strip("-").replace("--", "-") or "group"


def write_sheet(rows, group, capstone, path):
    L = ["# Integration map · " + group,
         "",
         "**Capstone:** " + capstone,
         "",
         "OQ Advanced AI for IT · Day 4 close. One row per system this capstone touches.",
         "Carried into Day 5: S26 (the MCP server), S27 and S28 (assembly), S29 (governance pack).",
         "",
         "## 1. The map", "",
         as_table(rows, MAP_VIEW),
         "",
         "## 2. The trust boundary", "",
         as_table(rows, TRUST_VIEW),
         "",
         "`trust` is whether anyone outside our team writes text that lands in that system.",
         "`egress` is whether content can leave through that hop in a shape the model chooses.",
         "",
         "## 3. Findings, from the four checks", ""]
    found = checks(rows)
    L += ["Nothing fired. Re-read §4's two questions before believing it."] if not found else \
         [f"{i}. **{rule}** — {text}" for i, (rule, text) in enumerate(found, 1)]
    L += ["", "## 4. The ask list", "",
          "| System | Who says yes | What we need | By when |", "|---|---|---|---|"]
    L += [f"| {s} | {o} | {n} |  |" for s, o, n in ask_list(rows)] or ["| | | | |"]
    L += ["", "## 5. The build list", ""]
    for stance in ("real", "mock", "out"):
        picked = [r for r in rows if r.get("day5") == stance]
        L.append(f"**{stance}** ({len(picked)})")
        L += [f"- {r['system']} — " + (r.get("blocker") if stance != "out" else "documented, not touched")
              for r in picked] or ["- none"]
        L.append("")
    L += ["## 6. Still open", "",
          "- Which channel of untrusted text into these systems have we not listed?",
          "- Who reads what we write, and what would they do at 03:00 if it were wrong?",
          "- Which of S24's five audit fields can we not produce for our writes today?",
          ""]
    path.write_text("\n".join(L), encoding="utf-8")
    return path


def write_blank(path):
    cols = "| " + " | ".join(MAP_VIEW) + " |"
    rule = "|" + "---|" * len(MAP_VIEW)
    tcols = "| " + " | ".join(TRUST_VIEW) + " |"
    trule = "|" + "---|" * len(TRUST_VIEW)
    L = ["# Integration mapping worksheet", "",
         "OQ Advanced AI for IT · Day 4 close · 15 minutes · one per group.",
         "One row per **system**, not per tool. Four rows is the right number.", "",
         "## The map", "", cols, rule] + ["| " + " | ".join([" "] * len(MAP_VIEW)) + " |"] * 5
    L += ["", "## The trust boundary", "", tcols, trule]
    L += ["| " + " | ".join([" "] * len(TRUST_VIEW)) + " |"] * 5
    L += ["", "## Legend", "",
          "- `direction` — read / write / read+write / call (a system holding none of our records)",
          "- `access` — have / requested / none. Today, not in Q2",
          "- `owner` — a person. A team has never approved anything",
          "- `authority` — which records, which states, how many",
          "- `trust` — untrusted if anyone outside our team writes text that lands there",
          "- `egress` — can content leave through this hop in a shape the model chooses",
          "- `session` — main, until a check makes you split one",
          "- `day5` — real / mock / out",
          "- `blocker` — the one thing that has to be true before this row is real", "",
          "## The four checks", "",
          "1. **Triangle** — untrusted in, something worth reading, a model-chosen way out, one session.",
          "2. **Unbounded write** — a write with no authority line. The prompt becomes the access control.",
          "3. **Fiction** — `real` on a system we do not have access to.",
          "4. **Read-only egress** — a read-only hop that can still carry content out.", ""]
    path.write_text("\n".join(L), encoding="utf-8")
    return path


filled = write_sheet(YOURS, GROUP, CAPSTONE, OUT / f"{slug(GROUP)}_integration_map.md")
blank = write_blank(OUT / "mapping_worksheet_blank.md")
print("wrote", filled.relative_to(ROOT))
print("wrote", blank.relative_to(ROOT))
print("\nBring the first one to Day 5, S27. Its ask list is page one of the governance pack in S29.")


## What to take away

- **A lab has two servers; an integration has owners.** Every row on that sheet is a person who can say no, a credential someone has to issue and a change window you do not control. None of that is in the model, and all of it decides whether this ships.
- **The mock is a design decision, so make it one.** Write what it stands in for and what has to be true before it is real. Then Thursday's demo is a staged integration rather than a thing that only works here.
- **Trust and egress are columns, not conversations.** Two words per system, filled in at the start, and you can see the channel before you build it. S25 took thirty minutes to show one; this sheet shows yours in a table.
- **Splitting a session is a field, not an architecture review.** When the triangle fires, one row moves. Do it while the sheet is still a sheet and it costs nothing.
- **The blocker column is what you say when someone asks what you need.** One line per system, written the week you started, is the difference between a project that waits and a project that escalates.
- **You now have four Day 4 artifacts that compose:** the rung (S20), the graph (S23), the blast radius (S24) and this map. Rung says how much autonomy, blast radius says what it may change, and the map says what it is allowed to reach in the first place.


## Facilitator notes

**The beats, and they are tight.**

| Min | What happens |
|---|---|
| 0:00 to 3:00 | §2 on screen. Read the four example rows out, and spend the time on the two cells that matter: `authority` on row one, `egress` on row three |
| 3:00 to 11:00 | Groups fill §3. Walk the room. Do not answer "is this a system?" — answer "who owns it?" and the question resolves itself |
| 11:00 to 14:00 | Run §4 and §5. Two groups read one finding each out loud, not the whole sheet |
| 14:00 to 15:00 | §6 writes the file. Say the handoff line and close the day |

**If a group has not settled its capstone.** Do not let them design one now. Give them the desk-and-documents pair from the example with their own department's names on it and have them fill `owner`, `access` and `blocker` for real. The skill is the sheet, not the use case, and S27 is where the use case lands.

**The three failure modes, in the order they show up.**

- **Rows that are wishes.** `access: have` because a colleague once got a token. Ask who called it, from what, this month.
- **Arguing about `trust`.** It eats four of the eight minutes. Rule it from the front: if anyone outside the team can put text in, it is untrusted, and say that almost everything at a plant qualifies.
- **Seven rows.** Two are the same system reached two ways. Merge them and move on.

**The check that will fire in most of the room is the triangle**, and the temptation is to treat it as a scary finding. It is not: it is the ordinary shape of an IT assistant, and it has a one-field fix. Show the fix, do not dwell.

**Handoff, said out loud as the day closes.** Tomorrow morning builds an MCP server in front of a mock ERP, as a class, and the reason it is worth eighty minutes is that everyone leaves able to do it again for the row on their own sheet. Bring the file.
